# Initialization

In [0]:
%run ../../utils/config

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

In [0]:
df.limit(2).display()

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.limit(2).display()

## Cleaning Dates

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8), None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8), None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8), None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

In [0]:
df.limit(2).display()

## Sales and Price Corrections

In [0]:

df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

In [0]:
df.limit(2).display()

## Filtering Valid Sales Transactions

In [0]:
# --- Business Rule: Only keep valid sales transactions ---
# We filter out any rows where the sales_amount is null or non-positive
df = df.filter((F.col("sls_sales").isNotNull()) & (F.col("sls_sales") > 0))

## Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
    

## Sanity checks of dataframe

In [0]:
df.limit(2).display()

# Writing Silver Table

In [0]:
from delta.tables import DeltaTable 

TARGET_TABLE = TABLES['crm_sales']
PK_COL = ["order_number", "product_number"]

if not spark.catalog.tableExists(TARGET_TABLE):
    df.write.format("delta").saveAsTable(TARGET_TABLE)
else:
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    merge_condition = " AND ".join([f"target.{col} = source.{col}" for col in PK_COL])
    (
        delta_target.alias("target")
        .merge(
            df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Collect MERGE Metrics

In [0]:
dt = DeltaTable.forName(spark, TARGET_TABLE)
metrics = dt.history(1).select("operationMetrics").collect()[0]
op_metrics = metrics["operationMetrics"]

# Sanity checks of silver table

In [0]:
spark.sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 5").display()

# Data Quality Checks

In [0]:
df_check = spark.table(TARGET_TABLE)
total_rows = df_check.count()
null_order = df_check.filter(F.col("order_number").isNull()).count()
duplicate_rows = total_rows - df_check.select(*PK_COL).distinct().count()
invalid_amount = df_check.filter((F.col("sales_amount").isNull()) | (F.col("sales_amount") <= 0)).count()

print(f"[QC] {TARGET_TABLE}")
print(f"  Total rows              : {total_rows}")
print(f"  Null order_number       : {null_order}")
print(f"  Duplicate (order+prod)  : {duplicate_rows}")
print(f"  Invalid sales_amount    : {invalid_amount}")

try:
    assert total_rows     > 0,  f"[QC FAILED] {TARGET_TABLE} is empty"
    assert null_order    == 0,  f"[QC FAILED] {null_order} null order_number values"
    assert duplicate_rows == 0, f"[QC FAILED] {duplicate_rows} duplicate (order_number, product_number) rows"
    assert invalid_amount == 0, f"[QC FAILED] {invalid_amount} rows with null or non-positive sales_amount"
    qc_status = "PASS"
    qc_message = "All checks passed"
    print("[QC PASSED]")
except AssertionError as e:
    qc_status = "FAIL"
    qc_message = str(e)
    print(f"[QC FAILED] {qc_message}")

## Write Audit Log

In [0]:
from datetime import datetime

row = [{
    "notebook_name": "silver_crm_sales_details",
    "target_table": TARGET_TABLE,
    "run_timestamp": datetime.now(),
    "rows_inserted": int(op_metrics.get("numTargetRowsInserted", 0)),
    "rows_updated": int(op_metrics.get("numTargetRowsUpdated", 0)),
    "rows_deleted": int(op_metrics.get("numTargetRowsDeleted", 0)),
    "qc_status": qc_status,
    "qc_message": qc_message
}]

df_audit = spark.createDataFrame(row)
df_audit.write.mode("append").format("delta").saveAsTable(TABLES["audit_log"])

## Sanity Check - Audit Log

In [0]:
spark.sql(f"SELECT * FROM {TABLES["audit_log"]}").display()